# DL predictions for the 4-detector thesis comparison (GPU)

Re-scores the LSTM + DistilBERT members of **all four** Yelp detectors on the full **17,891** corrected frontend reviews, on GPU, → `dl_predictions_all.csv`. The sklearn members are run **locally** by `merge_all_detectors.py`, which produces the honest 4-detector table.

* **baseline**    : `yelp_fake_lstm_onnx`, `yelp_fake_distilbert_onnx`
* **focal**       : `yelp_fake_lstm_focal_onnx`, `yelp_fake_distilbert_focal_onnx`
* **contrastive** : `yelp_fake_lstm_contrastive_onnx`, `yelp_fake_distilbert_contrastive_onnx`
* **modelb**      : `yelp_multimodal_lstm_full_onnx`, `yelp_multimodal_distilbert_full_onnx`

**Before running:** GPU runtime; upload `colab_dl_bundle_all.zip` to the session (or Drive + adjust the path).

In [ ]:
!pip -q install onnxruntime-gpu transformers joblib scikit-learn 2>/dev/null
import os, zipfile
BUNDLE = 'colab_dl_bundle_all.zip'   # adjust if uploaded to Drive
if not os.path.isdir('data'):
    with zipfile.ZipFile(BUNDLE) as z: z.extractall('data')
print('contents:', sorted(os.listdir('data')))

In [ ]:
import numpy as np, pandas as pd, json, glob, os
import onnxruntime as ort
print('providers:', ort.get_available_providers())
FEAT = ['rating','is_extreme','review_word_len','review_char_len','rating_dev_from_biz',
        'abs_rating_dev_from_biz','user_review_count','user_avg_rating','user_rating_std',
        'user_frac_positive','user_frac_extreme','user_is_singleton','user_max_reviews_per_day',
        'user_reviews_per_day','biz_review_count','biz_avg_rating','biz_rating_std']
df = pd.read_csv('data/yelp_frontend_multimodal.csv'); df['review_text']=df['review_text'].astype(str)
texts = df['review_text'].tolist(); beh_raw = df[FEAT].astype('float32').values
y = df['ground_truth_label'].astype(int).values
print('rows', len(df), '| fake rate', round(y.mean(),3))

In [ ]:
class KerasTok:
    def __init__(s, wi, filt, lower, nw): s.wi=wi; s._f=str.maketrans('','',filt); s.lower=lower; s.nw=nw
    @classmethod
    def load(cls, p):
        cfg=json.load(open(p,encoding='utf-8'))['config']; wi=cfg['word_index']
        if isinstance(wi,str): wi=json.loads(wi)
        return cls(wi, cfg.get('filters','!\"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'), cfg.get('lower',True), cfg.get('num_words'))
    def seqs(s, texts):
        out=[]
        for t in texts:
            if s.lower: t=t.lower()
            t=t.translate(s._f); seq=[]
            for w in t.split():
                i=s.wi.get(w)
                if i is not None and (s.nw is None or i<s.nw): seq.append(i)
            out.append(seq)
        return out

def pad(seqs, maxlen):
    a=np.zeros((len(seqs),maxlen),dtype=np.float32)
    for i,s in enumerate(seqs):
        s=s[:maxlen]; a[i,:len(s)]=s
    return a

def read_thr(d):
    for jf in glob.glob(os.path.join(d,'*.json')):
        try: o=json.load(open(jf))
        except Exception: continue
        if isinstance(o,dict) and 'threshold' in o: return float(o['threshold'])
    return 0.5

def score_dir(d, texts, beh_raw):
    sess=ort.InferenceSession(glob.glob(os.path.join(d,'*.onnx'))[0],
                              providers=['CUDAExecutionProvider','CPUExecutionProvider'])
    inames=[i.name for i in sess.get_inputs()]; thr=read_thr(d)
    def mlen(name, dflt):
        sh=[i.shape[1] for i in sess.get_inputs() if i.name==name][0]
        return sh if isinstance(sh,int) else dflt
    if 'input_layer' in inames:
        tok=KerasTok.load(glob.glob(os.path.join(d,'*_tokenizer.json'))[0])
        probs=sess.run(None,{'input_layer':pad(tok.seqs(texts), mlen('input_layer',150))})[0].reshape(-1)
    elif 'text_input' in inames:
        import joblib; sc=joblib.load(glob.glob(os.path.join(d,'*_scaler.joblib'))[0])
        tok=KerasTok.load(glob.glob(os.path.join(d,'*_tokenizer.json'))[0])
        beh=sc.transform(beh_raw).astype(np.float32)
        probs=sess.run(None,{'text_input':pad(tok.seqs(texts), mlen('text_input',250)),'beh_input':beh})[0].reshape(-1)
    else:
        from transformers import AutoTokenizer
        tok=AutoTokenizer.from_pretrained(d); is_mb=('beh' in inames); beh=None
        if is_mb:
            import joblib; sc=joblib.load(glob.glob(os.path.join(d,'*_scaler.joblib'))[0]); beh=sc.transform(beh_raw).astype(np.float32)
        out=[]; B=128
        for s in range(0,len(texts),B):
            enc=tok(texts[s:s+B],truncation=True,padding='max_length',max_length=256,return_tensors='np')
            feed={'input_ids':enc['input_ids'].astype('int64'),'attention_mask':enc['attention_mask'].astype('int64')}
            if is_mb: feed['beh']=beh[s:s+B]
            r=np.asarray(sess.run(None,feed)[0],dtype=np.float64)
            if r.ndim==2 and r.shape[1]==2:
                e=np.exp(r-r.max(1,keepdims=True)); p=(e/e.sum(1,keepdims=True))[:,1]
            else:
                p=1/(1+np.exp(-r.reshape(-1)))
            out.append(p)
        probs=np.concatenate(out)
    return probs, (probs>=thr).astype(int), thr

In [ ]:
ARMS = {
  'base_lstm':'data/yelp_fake_lstm_onnx',                 'base_db':'data/yelp_fake_distilbert_onnx',
  'focal_lstm':'data/yelp_fake_lstm_focal_onnx',          'focal_db':'data/yelp_fake_distilbert_focal_onnx',
  'contr_lstm':'data/yelp_fake_lstm_contrastive_onnx',    'contr_db':'data/yelp_fake_distilbert_contrastive_onnx',
  'mb_lstm':'data/yelp_multimodal_lstm_full_onnx',        'mb_db':'data/yelp_multimodal_distilbert_full_onnx',
}
from sklearn.metrics import f1_score
out = pd.DataFrame({'idx':np.arange(len(df)), 'ground_truth_label':y})
for col, d in ARMS.items():
    probs, labels, thr = score_dir(d, texts, beh_raw)
    out[col]=labels
    print(f'{col:<11} thr={thr:.2f}  macroF1={f1_score(y,labels,average="macro"):.3f}', flush=True)
out.to_csv('dl_predictions_all.csv', index=False)
print('\nwrote dl_predictions_all.csv', out.shape)
try:
    from google.colab import files; files.download('dl_predictions_all.csv')
except Exception as e:
    print('download manually from Files pane:', e)